# Multi-Model Research with E5 Embeddings and Optuna

This notebook explores NLI classification using pre-computed `multilingual-e5-small` embeddings.
We use Optuna for hyperparameter optimization across several models: XGBoost, LightGBM, and SVM.

In [ ]:
import json
from pathlib import Path

import joblib
import mlflow
import numpy as np
import pandas as pd
import optuna
from lightgbm import LGBMClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

from helpers.data import load_processed_data, load_sample_submission
from helpers.metrics import compute_all_metrics
from helpers.mlflow import log_common_context, log_metrics, log_model_artifacts, start_notebook_run
from helpers.submission import build_submission, save_submission

## Load Data and Embeddings

In [ ]:
PROCESSED_DIR = Path('../data/processed')
RAW_DIR = Path('../data/raw')
EMBEDDINGS_DIR = Path('../embeddings/e5')

train_df, val_df, test_df = load_processed_data(PROCESSED_DIR)
sample_submission = load_sample_submission(RAW_DIR / 'sample_submission.csv')

def load_embs(prefix):
    p = np.load(EMBEDDINGS_DIR / f'{prefix}_premise.npy')
    h = np.load(EMBEDDINGS_DIR / f'{prefix}_hypothesis.npy')
    return p, h

train_p, train_h = load_embs('train')
val_p, val_h = load_embs('val')
test_p, test_h = load_embs('test')

print(f'Data and embeddings loaded.')

## Feature Engineering

We combine premise and hypothesis embeddings using concatenation, absolute difference, and element-wise product.

In [ ]:
def prepare_features(p, h):
    return np.hstack([
        p, 
        h, 
        np.abs(p - h), 
        p * h
    ])

x_train = prepare_features(train_p, train_h)
x_val = prepare_features(val_p, val_h)
x_test = prepare_features(test_p, test_h)

y_train = train_df['label'].astype(int)
y_val = val_df['label'].astype(int)

print(f'Feature shape: {x_train.shape}')

## Optuna Optimization

In [ ]:
def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'random_state': 42,
        'eval_metric': 'mlogloss',
        'n_jobs': -1,
    }
    model = XGBClassifier(**params)
    model.fit(x_train, y_train)
    return model.score(x_val, y_val)

def objective_lgbm(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'random_state': 42,
        'verbosity': -1,
        'n_jobs': -1,
    }
    model = LGBMClassifier(**params)
    model.fit(x_train, y_train)
    return model.score(x_val, y_val)

def objective_svm(trial):
    params = {
        'C': trial.suggest_float('C', 0.1, 10.0, log=True),
        'kernel': trial.suggest_categorical('kernel', ['linear', 'rbf']),
        'probability': True,
        'random_state': 42,
    }
    model = SVC(**params)
    model.fit(x_train, y_train)
    return model.score(x_val, y_val)

## Run Experiments

In [ ]:
MODELS = {
    'xgboost': objective_xgb,
    'lightgbm': objective_lgbm,
    'svm': objective_svm,
}

results = []

for model_name, objective in MODELS.items():
    print(f'Optimizing {model_name}...')
    study = optuna.create_study(direction='maximize')
    # Limiting trials for research purposes
    study.optimize(objective, n_trials=10 if model_name != 'svm' else 5, show_progress_bar=True, n_jobs=1)
    
    best_params = study.best_params
    print(f'Best params for {model_name}: {best_params}')
    
    # Train final model with best params
    if model_name == 'xgboost':
        model = XGBClassifier(**best_params, random_state=42, eval_metric='mlogloss', n_jobs=-1)
    elif model_name == 'lightgbm':
        model = LGBMClassifier(**best_params, random_state=42, verbosity=-1, n_jobs=-1)
    else:
        model = SVC(**best_params, random_state=42, probability=True)
        
    model.fit(x_train, y_train)
    val_preds = model.predict(x_val)
    metrics = compute_all_metrics(val_df, val_preds)
    test_preds = model.predict(x_test)
    
    run_name = f'e5_optuna_{model_name}'
    submission = build_submission(sample_submission, test_preds)
    submission_path = Path('../submissions') / f'{run_name}.csv'
    save_submission(submission, submission_path)
    
    model_path = Path('../submissions') / f'{run_name}_model.joblib'
    joblib.dump(model, model_path)
    
    with start_notebook_run(
        run_name,
        tags={'model_family': model_name, 'features': 'e5_embeddings', 'optimized': 'optuna'}
    ):
        mlflow.log_params(best_params)
        log_metrics(metrics)
        log_common_context('../configs/data_split.yaml', '../data/processed/split_metadata.json')
        mlflow.log_artifact(str(submission_path), artifact_path='submissions')
        log_model_artifacts(
            artifacts={'model.joblib': model_path},
            metadata={'model_name': model_name, 'best_params': best_params},
            artifact_path='model'
        )
    
    results.append({
        'model': model_name,
        'accuracy': metrics['accuracy'],
        'f1_macro': metrics['f1_macro'],
        'best_params': best_params
    })

pd.DataFrame(results).sort_values('f1_macro', ascending=False)